In [17]:
import optuna
import numpy as np
import gymnasium as gym
from functools import partial
import torch.nn as nn
import os

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor, DummyVecEnv
from industrial_inventory_env import IndustrialInventoryEnv, generate_student_config

ROLL_NUMBER = "DA25M622"
student_config = generate_student_config(ROLL_NUMBER)
print(f"Student Config for {ROLL_NUMBER}: {student_config}")


def evaluate_cost(model, eval_seed=42, n_eval_episodes=15):
    """Evaluates the model and returns the average episode total cost (lower is better)."""
    eval_env = IndustrialInventoryEnv(
        student_config=student_config, 
        scenario_mode="random", 
        domain_randomization=True
    )
    episode_costs = []
    
    for ep in range(n_eval_episodes):
        obs, info = eval_env.reset(seed=eval_seed + ep)
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
        episode_costs.append(info["costs"]["episode_total"])
        
    eval_env.close()
    return float(np.mean(episode_costs))


def sample_params(trial: optuna.Trial) -> dict:
    """Sample A2C-specific hyperparameters (not PPO)."""
    # A2C uses these hyperparameters
    gamma = trial.suggest_categorical("gamma", [0.98, 0.99, 0.995])
    gae_lambda = trial.suggest_categorical("gae_lambda", [0.90, 0.95, 0.98])
    learning_rate = trial.suggest_float("learning_rate", 3e-5, 8e-4, log=True)
    ent_coef = trial.suggest_float("ent_coef", 1e-5, 1e-2, log=True)
    vf_coef = trial.suggest_float("vf_coef", 0.25, 0.75)
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.7)
    
    # A2C-specific: n_steps (rollout length)
    n_steps = trial.suggest_categorical("n_steps", [5, 10, 20, 32, 64, 128, 256])
    
    # Network architecture
    net_arch = trial.suggest_categorical("net_arch", ["small", "medium"])
    layers = [128, 128] if net_arch == "small" else [256, 256]
    
    policy_kwargs = dict(
        net_arch=dict(pi=layers, vf=layers),
        activation_fn=nn.ReLU
    )
    
    return {
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "learning_rate": learning_rate,
        "ent_coef": ent_coef,
        "vf_coef": vf_coef,
        "max_grad_norm": max_grad_norm,
        "n_steps": n_steps,
        "policy_kwargs": policy_kwargs,
    }


def objective(trial: optuna.Trial) -> float:
    params = sample_params(trial)
    
    env_fn = lambda: IndustrialInventoryEnv(
        student_config=student_config, 
        scenario_mode="random", 
        domain_randomization=True
    )
    
    vec_env = make_vec_env(env_id=env_fn, n_envs=4, seed=2026, vec_env_cls=DummyVecEnv)
    vec_env = VecMonitor(vec_env)
    
    model = A2C(
        policy="MultiInputPolicy",
        env=vec_env,
        verbose=0,
        seed=2026,
        **params
    )
    
    # Fast trial run for candidate screening
    try:
        model.learn(total_timesteps=60_000)
        mean_cost = evaluate_cost(model, eval_seed=101, n_eval_episodes=10)
    except Exception as e:
        vec_env.close()
        raise e
        
    vec_env.close()
    return mean_cost


# Create study and run optimization
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=2026),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

# 20 to 30 trials are typically sufficient to beat default configurations
study.optimize(objective, n_trials=25, timeout=3600)

print("Best Trial:")
print(f"  Mean Evaluated Cost: {study.best_trial.value:,.2f}")
print("  Params: ")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")


# Retrain with best hyperparams and save the model
best_params = study.best_trial.params
env_fn = lambda: IndustrialInventoryEnv(
    student_config=student_config, 
    scenario_mode="random", 
    domain_randomization=True
)

vec_env = make_vec_env(env_id=env_fn, n_envs=4, seed=2026, vec_env_cls=DummyVecEnv)
vec_env = VecMonitor(vec_env)

model = A2C(
    policy="MultiInputPolicy",
    env=vec_env,
    verbose=1,  # Set to 1 to see training progress
    seed=2026,
    **best_params
)

# Train final model with more timesteps for better performance
model.learn(total_timesteps=200_000)  # Increased from 60k for better convergence

# Save the best model
model.save("best_a2c_model")
vec_env.close()

print(f"Best model saved as 'best_a2c_model.zip'")
print(f"Best hyperparameters: {best_params}")


# Evaluate the saved model
# Load the saved model to verify
loaded_model = A2C.load("best_a2c_model")
mean_cost = evaluate_cost(loaded_model, eval_seed=2026, n_eval_episodes=20)
print(f"Final evaluation cost: {mean_cost:,.2f}")

[I 2026-09-02 22:59:16,293] A new study created in memory with name: no-name-0aa9fcbb-031f-4c6f-a192-8cf449f90982


Student Config for DA25M622: {'project_version': 'IITM-6002W-RL-Inventory-2026-v1', 'roll_number': 'DA25M622', 'variant_id': 'V022', 'demand_multiplier_profile': [1.0, 1.1, 0.9], 'initial_inventory_profile': [110, 100, 90], 'lead_time_delay_profile': [0.1, 0.0, 0.05], 'declared_ranges': {'demand_multiplier': [0.85, 1.15], 'initial_inventory': [80, 120], 'lead_time_delay_probability': [0.0, 0.1]}, 'config_fingerprint': '7d77fc79debf206a'}


/home/sohang/Projects/iit-madras-web-mtech-ai/trimester3/DA6002W_Online&ReinforcementLearning/ReinforcementLearning/RL_ProjectCompetition/RL_Student_Package_2026/.venv-sb3-torch213/lib/python3.12/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(
[I 2026-09-02 22:59:48,943] Trial 0 finished with value: 364564.0 and parameters: {'gamma': 0.995, 'gae_lambda': 0.98, 'learning_rate': 5.749952436638167e-05, 'ent_coef': 0.0054343242201941545, 'vf_coef': 0.5291250574329291, 'max_grad_norm': 0.6132917873903044, 'n_steps': 32, 'net_arch': 'medium'}. Best is trial 0 with value: 364564.0.
[I 2026-09-02 23:00:21,232] Trial 1 finished with value: 1860574.25 and parameters: {'gamma': 0.98, 'gae_lambda': 0.95, 'learning_rate': 0.0007175325553904467, 'ent_coef': 2.80221464989250

Best Trial:
  Mean Evaluated Cost: 164,147.00
  Params: 
    gamma: 0.995
    gae_lambda: 0.95
    learning_rate: 0.0001040837244840849
    ent_coef: 0.00020016991367226302
    vf_coef: 0.6335549492874142
    max_grad_norm: 0.5128069703757938
    n_steps: 256
    net_arch: medium


TypeError: A2C.__init__() got an unexpected keyword argument 'net_arch'